In [ ]:
import os, sys, subprocess

REPO_URL = "https://github.com/vietdai-bk/leo-haps-ue-datagen"
REPO_DIR = "/content/leo-haps-ue-datagen"

if os.path.isdir("leo_haps_ue_datagen"):
    REPO_DIR = os.getcwd()
elif os.path.isdir(os.path.join("..", "leo_haps_ue_datagen")):
    REPO_DIR = os.path.abspath("..")
elif os.path.isdir(os.path.join(REPO_DIR, ".git")):
    subprocess.run(["git", "-C", REPO_DIR, "pull", "--ff-only"], check=False)  # cập nhật bản đã clone
elif not os.path.isdir(REPO_DIR):
    if REPO_URL and "<your-username>" not in REPO_URL:
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    else:
        from google.colab import files
        up = files.upload()
        zip_name = next(iter(up))
        subprocess.run(["unzip", "-q", "-o", zip_name, "-d", "/content"], check=True)
        if not os.path.isdir(REPO_DIR):
            REPO_DIR = os.path.dirname(next(p for p, d, f in os.walk("/content") if "leo_haps_ue_datagen" in d) + "/leo_haps_ue_datagen")

if not os.path.isdir(os.path.join(REPO_DIR, "leo_haps_ue_datagen")):
    raise FileNotFoundError(f"{REPO_DIR} không chứa package leo_haps_ue_datagen: repo trống, chưa push code, "
                            "hoặc REPO_URL sai. Kiểm tra lại repo trên GitHub (phải public).")
os.chdir(REPO_DIR)
sys.path.insert(0, REPO_DIR)
print("Repo:", REPO_DIR)
print(sorted(os.listdir(REPO_DIR)))

In [ ]:
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
import numpy, scipy, sgp4, pymap3d, pyarrow
print("numpy", numpy.__version__, "| sgp4", sgp4.__version__, "| pymap3d", pymap3d.__version__, "| pyarrow", pyarrow.__version__)

In [ ]:
RUN_TESTS = True
if RUN_TESTS:
    r = subprocess.run([sys.executable, "-m", "pytest", "tests", "-q", "-W", "ignore"], capture_output=True, text=True)
    print(r.stdout[-1500:] or r.stderr[-1500:])

In [ ]:
from leo_haps_ue_datagen import build_dataset as B

quick_cfg = dict(B.DEFAULTS, duration_s=300, n_ue=10)
tables, meta, extra = B.build(quick_cfg, "output/quick")
B.plot(tables, extra, "output/quick")
print("rows:", meta["row_counts"])
print("validation ok:", meta["validation"]["ok"], "| build", meta["build_seconds"], "s")

In [ ]:
cfg = dict(B.DEFAULTS)
cfg.update(
    orbit="data/STARLINK_20250810.txt",
    start="2025-08-10T06:00:00",
    duration_s=3600,
    dt_s=1.0,
    n_ue=50,
    ue_radius_m=10e3,
    p_critical=0.2,
    seed=0,
    haps=dict(lat=35.6762, lon=139.6503, alt_m=20000.0),
    fmt="parquet",      # hoặc "csv" (mở được bằng Excel, nhưng nặng hơn ~3.7 lần)
)
OUT = "output/run"

assert cfg["duration_s"] * cfg["n_ue"] <= 180_000 or input("Cấu hình lớn, có thể hết RAM. Tiếp tục? [y/N] ") == "y"
tables, meta, extra = B.build(cfg, OUT)
B.plot(tables, extra, OUT)
del tables, extra
print("rows:", meta["row_counts"])
print("validation:", {k: v for k, v in meta["validation"].items() if not k.endswith("counts")})
print("build", meta["build_seconds"], "s ->", OUT)

In [ ]:
from IPython.display import Image, display
display(Image(filename=os.path.join(OUT, "validation.png")))

In [ ]:
import pandas as pd, json

sat_haps = B.read_table(OUT, "sat_haps").to_pandas()
haps_ue  = B.read_table(OUT, "haps_ue").to_pandas()
ue       = B.read_table(OUT, "ue").to_pandas()
# sat_ue rất lớn (~14 triệu dòng): chỉ đọc vài cột, user_type dạng category để tiết kiệm RAM
sat_ue   = B.read_table(OUT, "sat_ue",
                        columns=["t_idx", "sat_id", "ue_id", "user_type", "elev_deg", "snr_sat_ue_dB",
                                 "available", "rst_sat_ue_s", "rst_status", "isac_snr_dB"]
                        ).to_pandas(categories=["user_type"])
meta = json.load(open(os.path.join(OUT, "metadata.json")))

for name, df in [("sat_haps", sat_haps), ("sat_ue", sat_ue), ("haps_ue", haps_ue), ("ue", ue)]:
    print(f"{name:9s} {df.shape}")
sat_haps.head()

In [ ]:
av = sat_haps[sat_haps.available]
print("Số vệ tinh khả dụng cho HAPS mỗi giây: trung bình %.1f, min %d, max %d" %
      tuple(av.groupby("t_idx").size().agg(["mean", "min", "max"])))
print("FSO capacity [Gbps]:", (av.fso_capacity_bps / 1e9).describe().round(2).to_dict())
print("RST sat–HAPS [s] (đang phục vụ):", av.rst_sat_haps_s.dropna().describe().round(1).to_dict())
print("Tỉ lệ dòng censored:", round((av.rst_status == 2).mean(), 4))
print()
print(haps_ue.groupby("user_type")[["snr_haps_ue_dB", "isac_snr_dB"]].describe().round(2).T)
print()
print(sat_ue[sat_ue.available].groupby("user_type").snr_sat_ue_dB.describe().round(2))
print()
print("Trạng thái di chuyển:", ue.state.value_counts(normalize=True).round(3).to_dict())

In [ ]:
# Trực quan hóa: vệ tinh HAPS chọn theo chính sách ví dụ "giữ tới khi mất, rồi chọn RST lớn nhất"
%matplotlib inline
import matplotlib.pyplot as plt
from leo_haps_ue_datagen import viz

d = viz.load(OUT)
sel = viz.greedy_selection(d["sat_haps"])
print("Số lần handover:", len(viz.handovers(sel)), "| thời gian HAPS không có vệ tinh:", int((sel < 0).sum()), "s")

In [ ]:
# Sky plot động nhìn từ HAPS (tâm = thiên đỉnh, vòng xanh = ngưỡng 30°, sao đỏ = vệ tinh đang dùng)
viz.show(viz.sky_animation(d, sel, step=10))

In [ ]:
# UE di chuyển theo chuỗi Markov (đỏ = critical, xanh = normal; o / ^ / s = đứng yên / đi bộ / xe)
viz.show(viz.ue_animation(d, step=10))

In [ ]:
# Quỹ đạo 3D tương tác (kéo để xoay, nút ▶ để chạy; đường đỏ = liên kết HAPS–vệ tinh)
viz.orbit_3d(d, sel, step=20).show()

In [ ]:
# Timeline khả dụng của vệ tinh: chỉnh t0 / window_s để xem đoạn khác
fig = viz.timeline(d, sel, t0=0, window_s=600)
display(fig)
plt.close(fig)

In [ ]:
# (Tùy chọn) lưu animation thành GIF
SAVE_GIF = False
if SAVE_GIF:
    viz.save(viz.sky_animation(d, sel, step=20), os.path.join(OUT, "sky.gif"))
    viz.save(viz.ue_animation(d, step=20), os.path.join(OUT, "ue.gif"))
    print("Đã lưu", os.path.join(OUT, "sky.gif"), os.path.join(OUT, "ue.gif"))

In [ ]:
import numpy as np

p_dBm = 30.0
snr_30 = p_dBm + 10 * np.log10(haps_ue.channel_gain) - haps_ue.noise_power_dBm
print("SNR @30 dBm, chênh so với @p_max (phải đúng -10 dB):", float((snr_30 - haps_ue.snr_haps_ue_dB).mean()))

n_ue = haps_ue.groupby("t_idx").ue_id.transform("count")
B_i = haps_ue.bandwidth_hz / n_ue
p_i = 10 ** ((haps_ue.p_max_dBm - 30) / 10) / n_ue
N0 = 10 ** ((haps_ue.noise_power_dBm - 30) / 10) / haps_ue.bandwidth_hz
rate = B_i * np.log2(1 + p_i * haps_ue.channel_gain / (N0 * B_i))
min_rate = haps_ue[["t_idx", "ue_id"]].merge(ue[["t_idx", "ue_id", "min_rate_bps"]], on=["t_idx", "ue_id"], how="left")
qos_ok = rate.values >= min_rate.min_rate_bps.values
print("Tỉ lệ (UE, t) đạt min_rate khi chia đều:", round(float(qos_ok.mean()), 4))

In [ ]:
SAVE_TO_DRIVE = False
DOWNLOAD_ZIP = False

try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB and SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    dst = "/content/drive/MyDrive/leo_haps_ue_datagen_" + os.path.basename(OUT)
    import shutil
    shutil.copytree(OUT, dst, dirs_exist_ok=True)   # chạy lại không tạo thư mục lồng nhau
    print("Đã lưu vào", dst)

if IN_COLAB and DOWNLOAD_ZIP:
    from google.colab import files
    zip_path = f"/content/{os.path.basename(OUT)}.zip"
    subprocess.run(["zip", "-q", "-r", zip_path, OUT], check=True)
    files.download(zip_path)